# 04 — RAG Evaluation with RAGAS

**Lab 05 · RAG with LangChain**

---

Running a RAG pipeline is not enough — you need to **measure** whether it
actually works. RAGAS provides four complementary metrics that together give
a complete picture of pipeline quality:

| Metric | Question answered | What it catches |
|---|---|---|
| **Context Precision** | Are all retrieved chunks relevant? | Noisy retrieval |
| **Context Recall** | Does the context contain all facts needed? | Missing chunks |
| **Faithfulness** | Is the answer grounded in the context? | Hallucination |
| **Answer Relevancy** | Does the answer address the question? | Off-topic answers |

> **Prerequisites**: API key for at least one LLM provider in `.env`
> (same as notebook 03). RAGAS uses the LLM to score Faithfulness and
> Answer Relevancy.

This notebook is **standalone** — no dependency on other lab modules.

## 0 — Setup: pipeline and knowledge base

In [ ]:
import tempfile
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from sentence_transformers import CrossEncoder

load_dotenv(find_dotenv())

# ---------------------------------------------------------------------------
# LLM
# ---------------------------------------------------------------------------

MODEL_ID = "anthropic:claude-sonnet-4-6"
llm = init_chat_model(MODEL_ID)
print(f"LLM: {MODEL_ID}")

# ---------------------------------------------------------------------------
# Knowledge base (same passages as notebook 03)
# ---------------------------------------------------------------------------

PASSAGES = [
    (
        "transformer_architecture",
        "The Transformer architecture, introduced in 'Attention Is All You Need'"
        " (2017), replaced recurrent networks with a self-attention mechanism."
        " Self-attention allows each token to attend to every other token in the"
        " sequence simultaneously, enabling full parallelism during training."
        " The encoder-decoder design uses multi-head attention to capture"
        " different semantic relationships at multiple scales.",
    ),
    (
        "rag_overview",
        "Retrieval-Augmented Generation (RAG) combines a retrieval system with"
        " a generative language model. At query time, relevant passages are"
        " fetched from a vector store and injected into the LLM prompt as"
        " context. This grounds the model's response in factual documents,"
        " reducing hallucination and enabling knowledge updates without"
        " retraining the model.",
    ),
    (
        "cross_encoder",
        "A CrossEncoder takes a (query, passage) pair as a single input and"
        " produces a relevance score using full bidirectional attention across"
        " both texts. This is more accurate than bi-encoder similarity but"
        " much slower — O(n) inference calls per query. In practice,"
        " CrossEncoders are used as a reranker on a small candidate set"
        " retrieved by a fast bi-encoder (e.g. FAISS or ChromaDB).",
    ),
    (
        "query_expansion",
        "Query expansion generates alternative phrasings of a user question"
        " to improve retrieval recall. A single query may not match the exact"
        " vocabulary used in relevant documents. By generating 2-3 variants"
        " using an LLM and merging their retrieval results, the pipeline"
        " surfaces passages that would be missed by the original query alone.",
    ),
    (
        "lcel_chains",
        "LangChain Expression Language (LCEL) uses the pipe operator (|) to"
        " compose runnables: prompts, models, parsers, and retrievers. A chain"
        " like prompt | llm | StrOutputParser() is lazy — it builds a"
        " computation graph without executing it. LCEL chains support"
        " streaming, async, and batching transparently, and can be deployed"
        " with LangServe.",
    ),
]

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)

docs = [
    Document(page_content=text, metadata={"topic": topic})
    for topic, text in PASSAGES
]
chunks = splitter.split_documents(docs)

DB_DIR = Path(tempfile.mkdtemp()) / "chroma_nb04"
vectorstore = Chroma(
    collection_name="rag_eval",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR),
)
vectorstore.add_documents(chunks)
print(f"Indexed {vectorstore._collection.count()} chunks")

In [ ]:
# RAG pipeline (same as notebook 03)

class QueryExpansion(BaseModel):
    questions: list[str] = Field(
        description="Alternative phrasings of the original question"
    )


EXPANSION_PROMPT = ChatPromptTemplate.from_template(
    "Generate {n} alternative phrasings of the following question to improve"
    " document retrieval. Return only the alternatives, not the original.\n\n"
    "Original question: {question}"
)

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using ONLY the context"
    " provided below. If the context does not contain enough information, say"
    ' "I don\'t have enough information to answer this question."\n\n'
    "Context:\n{context}\n\n"
    "Question: {question}\n\nAnswer:"
)

rag_chain = RAG_PROMPT | llm | StrOutputParser()
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def expand_query(question: str, n: int = 2) -> list[str]:
    structured_llm = llm.with_structured_output(QueryExpansion)
    result: QueryExpansion = structured_llm.invoke(
        EXPANSION_PROMPT.format_messages(question=question, n=n)
    )
    return [question] + result.questions


def retrieve(questions: list[str], top_k: int = 5) -> list[Document]:
    seen: set[str] = set()
    all_docs: list[Document] = []
    for q in questions:
        for doc in vectorstore.similarity_search(q, k=top_k):
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)
    return all_docs


def rerank_docs(
    query: str, documents: list[Document], top_k: int = 3
) -> list[Document]:
    if not documents:
        return []
    pairs = [(query, doc.page_content) for doc in documents]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(scores, documents, strict=True), key=lambda x: x[0], reverse=True
    )
    return [doc for _, doc in ranked][:top_k]


def rag_answer(question: str) -> dict:
    questions = expand_query(question, n=2)
    retrieved = retrieve(questions)
    if not retrieved:
        return {"answer": "I don't have enough information.", "contexts": []}
    top_docs = rerank_docs(question, retrieved)
    ctx = "\n\n---\n\n".join(doc.page_content for doc in top_docs)
    answer = rag_chain.invoke({"context": ctx, "question": question})
    return {"answer": answer, "contexts": [doc.page_content for doc in top_docs]}


print("Pipeline ready.")

## 1 — Ground-truth dataset

A RAGAS evaluation requires a dataset of `(question, ground_truth, contexts)`
triples:

- **question** — the user query
- **ground_truth** — the ideal answer, written by a human expert
- **contexts** — the passages retrieved by the pipeline for that question

The quality of your ground-truth directly determines the meaning of the scores:
a sloppy ground-truth produces misleading metrics.

In [ ]:
# Hand-written Q&A pairs grounded in the knowledge base above
GROUND_TRUTH_QA = [
    {
        "question": (
            "What is Retrieval-Augmented Generation and why does it"
            " reduce hallucination?"
        ),
        "ground_truth": (
            "Retrieval-Augmented Generation (RAG) combines a retrieval"
            " system with a generative language model. At query time,"
            " relevant passages are fetched from a vector store and"
            " injected into the LLM prompt as context. This grounds the"
            " model's response in factual documents, reducing hallucination"
            " because the model is constrained to retrieved content rather"
            " than relying solely on its parametric knowledge."
        ),
    },
    {
        "question": (
            "Why is a CrossEncoder more accurate than a bi-encoder"
            " for reranking?"
        ),
        "ground_truth": (
            "A CrossEncoder encodes the query and the passage jointly in a"
            " single forward pass, allowing every query token to attend to"
            " every passage token. This full bidirectional attention produces"
            " a more accurate relevance score than a bi-encoder, which"
            " encodes query and document independently and compares them"
            " via dot product."
        ),
    },
    {
        "question": "What is the purpose of query expansion in RAG?",
        "ground_truth": (
            "Query expansion generates alternative phrasings of the user"
            " question to improve retrieval recall. A single query may not"
            " match the vocabulary used in relevant documents. By generating"
            " 2-3 variants with an LLM and merging their retrieval results,"
            " the pipeline surfaces passages that the original query"
            " would have missed."
        ),
    },
    {
        "question": "How does LCEL compose a generation chain in LangChain?",
        "ground_truth": (
            "LCEL uses the pipe operator (|) to compose Runnables — prompts,"
            " models, parsers, and retrievers. A chain like"
            " prompt | llm | StrOutputParser() is lazy: it builds a"
            " computation graph without executing it. The chain runs when"
            " .invoke() is called and supports streaming, async, and batching."
        ),
    },
    {
        "question": (
            "What innovation did the Transformer introduce over"
            " recurrent networks?"
        ),
        "ground_truth": (
            "The Transformer replaced recurrent networks with a self-attention"
            " mechanism. Self-attention allows each token to attend to every"
            " other token simultaneously, enabling full parallelism during"
            " training instead of sequential processing. The encoder-decoder"
            " design uses multi-head attention to capture different semantic"
            " relationships at multiple scales."
        ),
    },
]

print(f"Ground-truth dataset: {len(GROUND_TRUTH_QA)} Q&A pairs")

## 2 — Run the pipeline on all questions

Generate answers and collect the retrieved contexts for each question.
This produces the `answer` and `contexts` columns needed by RAGAS.

In [ ]:
eval_rows = []

for i, qa in enumerate(GROUND_TRUTH_QA):
    print(f"[{i+1}/{len(GROUND_TRUTH_QA)}] {qa['question'][:60]}...")
    result = rag_answer(qa["question"])
    eval_rows.append(
        {
            "question": qa["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": qa["ground_truth"],
        }
    )
    n_ctx = len(result["contexts"])
    n_chars = len(result["answer"])
    print(f"         → {n_ctx} contexts, {n_chars} chars\n")

print("All answers generated.")

## 3 — Evaluate with RAGAS

RAGAS expects a HuggingFace `Dataset` with the four columns above.
The `evaluate()` call runs each metric in turn and returns a result object
with per-question scores and an aggregate mean.

In [ ]:
import warnings

import anthropic
from datasets import Dataset
from ragas import evaluate
from ragas.llms import llm_factory

# ragas.metrics singletons and LangchainEmbeddingsWrapper are deprecated in
# 0.4.x but still functional — suppress their warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics import (
        answer_relevancy,
        context_precision,
        context_recall,
        faithfulness,
    )
    # Instantiate inside the block — the warning fires at instantiation time
    ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# LLM — native RAGAS provider; pop top_p because Anthropic rejects requests
# that specify both temperature and top_p simultaneously
ragas_llm = llm_factory(
    "claude-sonnet-4-6", provider="anthropic", client=anthropic.Anthropic()
)
ragas_llm.model_args.pop("top_p", None)

faithfulness.llm = ragas_llm
context_precision.llm = ragas_llm
context_recall.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings

dataset = Dataset.from_list(eval_rows)

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

print(result)

## 4 — Visualise results

In [ ]:
import matplotlib.pyplot as plt

df = result.to_pandas()

# --- Aggregate bar chart ---
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
means = df[metrics].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].bar(
    [m.replace("_", "\n") for m in metrics],
    means.values,
    color=["#4C9BE8", "#50C878", "#E86B4C", "#F5A623"],
)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Score (0–1)")
axes[0].set_title("RAGAS — aggregate scores")
axes[0].axhline(0.8, color="grey", linestyle="--", linewidth=0.8, label="0.8 target")
axes[0].legend(fontsize=8)
for bar, val in zip(bars, means.values, strict=True):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.02,
        f"{val:.2f}",
        ha="center",
        fontsize=9,
    )

# --- Per-question heatmap ---
short_q = [f"Q{i+1}" for i in range(len(df))]
heat_data = df[metrics].values
im = axes[1].imshow(heat_data.T, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
axes[1].set_xticks(range(len(short_q)))
axes[1].set_xticklabels(short_q)
axes[1].set_yticks(range(len(metrics)))
axes[1].set_yticklabels([m.replace("_", "\n") for m in metrics], fontsize=8)
axes[1].set_title("RAGAS — per-question scores")
plt.colorbar(im, ax=axes[1])
for i in range(len(metrics)):
    for j in range(len(df)):
        score = f"{heat_data[j, i]:.2f}"
        axes[1].text(j, i, score, ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Per-question detail table
# RAGAS 0.4.x renames dataset columns: question→user_input, contexts→retrieved_contexts
display_df = df[["user_input"] + metrics].copy()
display_df["user_input"] = display_df["user_input"].str[:55] + "..."
display_df = display_df.rename(columns={
    "user_input": "Question",
    "faithfulness": "Faith.",
    "answer_relevancy": "Ans.Rel.",
    "context_precision": "Ctx.Prec.",
    "context_recall": "Ctx.Rec.",
})
display_df.index = [f"Q{i+1}" for i in range(len(display_df))]
display_df.round(3)

## 5 — Interpreting the results

| Score range | Interpretation | Action |
|---|---|---|
| `≥ 0.8` | Good | No immediate action needed |
| `0.6–0.8` | Acceptable | Monitor; consider targeted improvements |
| `< 0.6` | Problematic | Investigate root cause (see below) |

**Diagnosis by metric:**

- **Low Faithfulness** → the LLM adds information not in the context. Tighten
  the prompt: "Answer using ONLY the context provided."
- **Low Answer Relevancy** → the answer drifts off-topic. Check whether the
  retrieved context is relevant, or whether the prompt needs more guidance.
- **Low Context Precision** → retrieval returns noisy chunks. Reduce `top_k`,
  increase reranking aggressiveness, or improve chunking.
- **Low Context Recall** → relevant passages are missed entirely. Increase
  `top_k`, improve chunking overlap, or add query expansion variants.

## 6 — Key takeaways

RAGAS turns qualitative impressions ("the answers seem good") into quantitative
signals that can be tracked over time and across pipeline configurations.

In a production RAG system you would:
1. Build a ground-truth dataset from real user queries and expert-validated answers
2. Run RAGAS after every pipeline change (chunking strategy, embedding model, reranker)
3. Treat it like a test suite — failing metrics block the deployment

**What comes next:**
- `README.md` — architecture diagram, setup instructions, sample_docs sources